In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

PyTorch version: 2.11.0+cu130
GPU available: True
GPU name: Tesla T4
GPU memory: 14.56 GB


In [ ]:
%pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Downloading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading AI model onto the T4 GPU...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("SUCCESS! Your AI model is loaded.")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

SUCCESS! Your AI model is loaded.


In [ ]:
messages = [
    {
        "role": "user",
        "content": "Hello! You are my personal AI assistant. Introduce yourself and explain what you can help me with."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("AI is thinking...")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.8
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("\nYOUR AI ANSWERS:\n")
print(answer)

AI is thinking...

YOUR AI ANSWERS:

Hello! I'm Qwen, your personal AI assistant. I'm here to help you with a wide range of tasks and questions. Whether you need assistance with writing, coding, learning, or just want to chat, I'm here to support you. I can help with:

- Answering questions on various topics
- Writing essays, stories, or emails
- Coding and debugging
- Learning new skills or languages
- Creating content like poems or scripts
- Providing recommendations or suggestions

Feel free to ask me anything, and I'll do my best to assist you! How can I help you today?
